In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [2]:
data_2022 = pd.read_excel("supply_chain_data.xlsx", sheet_name = 'Demand_2022', header=1, usecols = "A:G")
data_2022 = data_2022[data_2022["Date"].astype(str).str.match(r"\d{4}-\d{2}-\d{2}")].copy()
data_2023 = pd.read_excel("supply_chain_data.xlsx", sheet_name = 'Demand_2023', header=1, usecols = "A:G")
data_2023 = data_2023[data_2023["Date"].astype(str).str.match(r"\d{4}-\d{2}-\d{2}")].copy()
data_2024 = pd.read_excel("supply_chain_data.xlsx", sheet_name = 'Demand_2024', header=1, usecols = "A:G")
data_2024 = data_2024[data_2024["Date"].astype(str).str.match(r"\d{4}-\d{2}-\d{2}")].copy()

data = pd.concat([data_2022, data_2023, data_2024], ignore_index=True)
data = data[["Date", "Daily Demand (Units)"]]
data["Date"] = pd.to_datetime(data["Date"])
data["Daily Demand (Units)"] = pd.to_numeric(data["Daily Demand (Units)"], errors='coerce')
df = data[['Date', 'Daily Demand (Units)']]
df.set_index('Date', inplace=True)
df.index.freq = pd.infer_freq(df.index)

monthly = df["Daily Demand (Units)"].resample("M").mean()
monthly.index = monthly.index.to_period("M").to_timestamp()

In [3]:
model  = ExponentialSmoothing(monthly, trend="add", seasonal="add",
                               seasonal_periods=12).fit(optimized=True)

# ── Forecast next month ──
forecast       = model.forecast(1)
forecast_date  = forecast.index[0]
forecast_mean  = forecast.iloc[0]

# ── Standard deviation from residuals ──
residuals      = monthly.values - model.fittedvalues.values
forecast_std   = np.std(residuals, ddof=1)

print(f"Forecast Month : {forecast_date.strftime('%B %Y')}")
print(f"Mean  (μ)      : {forecast_mean:.4f} units/day")
print(f"Std   (σ)      : {forecast_std:.4f}  units/day")

Forecast Month : January 2025
Mean  (μ)      : 230.5913 units/day
Std   (σ)      : 2.8049  units/day


In [5]:
forecast

2025-01-01    230.591286
Freq: MS, dtype: float64